In [ ]:
#source activate newEnv
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(dplyr))
suppressMessages(library(sctransform))
suppressMessages(library(reticulate))
suppressMessages(library(future))
suppressMessages(library(plyr))
suppressMessages(library('Biobase'))
suppressMessages(library(gplots))
suppressMessages(library('hdf5r'))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(BSgenome.Hsapiens.UCSC.hg38))
suppressMessages(library(BiocParallel))
suppressMessages(library(cowplot))
suppressMessages(library(GenomeInfoDb))
suppressMessages(library(ArchR))
library(stringr)
library(Matrix)
library(SummarizedExperiment)
library(matrixStats)
library(readr)
library(GenomicRanges)
library(magrittr)
library(edgeR)
library(Seurat)
library(BSgenome.Hsapiens.UCSC.hg38)

# FINAL OUTDIR

In [ ]:
### directory to write files to
fin_dir <- '/nfs/lab/projects/nPOD/downstream_files/ATAC/pseudobulk_nPODids/fixed_peak/'


# Make fixed length peak file and cell type specific fixed peak files

In [ ]:
#### various functions needed from the greenleaf lab github
extendedPeakSet <- function(df, BSgenome = NULL, extend = 250, blacklist = NULL, nSummits = 100000){
  #Helper Functions
  readSummits <- function(file){
    df <- suppressMessages(data.frame(readr::read_tsv(file, col_names = c("chr","start","end","name","score"))))
    df <- df[,c(1,2,3,5)] #do not keep name column it can make the size really large
    return(GenomicRanges::makeGRangesFromDataFrame(df=df,keep.extra.columns = TRUE,starts.in.df.are.0based = TRUE))
  }
  nonOverlappingGRanges <- function(gr, by = "score", decreasing = TRUE, verbose = FALSE){
    stopifnot(by %in% colnames(mcols(gr)))
    clusterGRanges <- function(gr, filter = TRUE, by = "score", decreasing = TRUE){
      gr <- sort(sortSeqlevels(gr))
      r <- GenomicRanges::reduce(gr, min.gapwidth=0L, ignore.strand=TRUE)
      o <- findOverlaps(gr,r)
      mcols(gr)$cluster <- subjectHits(o)
      gr <- gr[order(mcols(gr)[,by], decreasing = decreasing),]
      gr <- gr[!duplicated(mcols(gr)$cluster),]
      gr <- sort(sortSeqlevels(gr))
      mcols(gr)$cluster <- NULL
      return(gr)
    }
    if(verbose){
      message("Converging", appendLF = FALSE)
    }
    i <-  0
    gr_converge <- gr
    while(length(gr_converge) > 0){
      if(verbose){
        message(".", appendLF = FALSE)
      }
      i <-  i + 1
      gr_selected <- clusterGRanges(gr = gr_converge, filter = TRUE, by = by, decreasing = decreasing)
      gr_converge <- subsetByOverlaps(gr_converge ,gr_selected, invert=TRUE) #blacklist selected gr
      if(i == 1){ #if i=1 then set gr_all to clustered
        gr_all <- gr_selected
      }else{
        gr_all <- c(gr_all, gr_selected)
      } 
    }
    if(verbose){
      message("\nSelected ", length(gr_all), " from ", length(gr))
    }
    gr_all <- sort(sortSeqlevels(gr_all))
    return(gr_all)
  }
  #Check-------
  stopifnot(extend > 0)
  stopifnot("samples" %in% colnames(df))
  stopifnot("groups" %in% colnames(df))
  stopifnot("summits" %in% colnames(df))
  stopifnot(!is.null(BSgenome))
  stopifnot(all(apply(df,1,function(x){file.exists(paste0(x[3]))})))
  #------------
  #Deal with blacklist
  if(is.null(blacklist)){
    blacklist <- GRanges()
  }else if(is.character(blacklist)){
    blacklist <- rtracklayer::import.bed(blacklist)
  }
  stopifnot(inherits(blacklist,"GenomicRanges"))
  #------------
  #Time to do stuff
  chromSizes <- GRanges(names(seqlengths(BSgenome)), IRanges(1, seqlengths(BSgenome)))
  chromSizes <- GenomeInfoDb::keepStandardChromosomes(chromSizes, pruning.mode = "coarse")
  groups <- unique(df$groups)
  groupGRList <- GenomicRanges::GRangesList(lapply(seq_along(groups), function(i){
      df_group = df[which(df$groups==groups[i]),]
      grList <- GenomicRanges::GRangesList(lapply(paste0(df_group$summits), function(x){
        extended_summits <- readSummits(x) %>%
          resize(., width = 2 * extend + 1, fix = "center") %>%     
          subsetByOverlaps(.,chromSizes,type="within") %>%
          subsetByOverlaps(.,blacklist,invert=TRUE) %>%
          nonOverlappingGRanges(., by="score", decreasing=TRUE)
        extended_summits <- extended_summits[order(extended_summits$score,decreasing=TRUE)]
        if(!is.null(nSummits)){
          extended_summits <- head(extended_summits, nSummits)
        }
        mcols(extended_summits)$scoreQuantile <- trunc(rank(mcols(extended_summits)$score))/length(mcols(extended_summits)$score)
        extended_summits
      }))
      #Non Overlapping
      grNonOverlapping <- nonOverlappingGRanges(unlist(grList), by = "scoreQuantile", decreasing = TRUE)
      #Free Up Memory
      remove(grList)
      gc()
      grNonOverlapping
    }))
  grFinal <- nonOverlappingGRanges(unlist(groupGRList), by = "scoreQuantile", decreasing = TRUE)
  grFinal <- sort(sortSeqlevels(grFinal))
  return(grFinal)
}

groupSums <- function(mat, groups = NULL, na.rm = TRUE, sparse = FALSE){
    stopifnot(!is.null(groups))
    stopifnot(length(groups) == ncol(mat))
    gm <- lapply(unique(groups), function(x) {
        if (sparse) {
            Matrix::rowSums(mat[, which(groups == x), drop = F], na.rm = na.rm)
        }else {
            rowSums(mat[, which(groups == x), drop = F], na.rm = na.rm)
        }
    }) %>% Reduce("cbind", .)
    colnames(gm) <- unique(groups)
    return(gm)
}


countInsertions <- function(query, fragments, by = "RG"){
  #Count By Fragments Insertions
  inserts <- c(
    GRanges(seqnames = seqnames(fragments), ranges = IRanges(start(fragments), start(fragments)), RG = mcols(fragments)[,by]),
    GRanges(seqnames = seqnames(fragments), ranges = IRanges(end(fragments), end(fragments)), RG = mcols(fragments)[,by])
  )
  by <- "RG"
  overlapDF <- DataFrame(findOverlaps(query, inserts, ignore.strand = TRUE, maxgap=-1L, minoverlap=0L, type = "any"))
  overlapDF$name <- mcols(inserts)[overlapDF[, 2], by]
  overlapTDF <- transform(overlapDF, id = match(name, unique(name)))
  #Calculate Overlap Stats
  inPeaks <- table(overlapDF$name)
  total <- table(mcols(inserts)[, by])
  total <- total[names(inPeaks)]
  frip <- inPeaks / total
  #Summarize
  sparseM <- Matrix::sparseMatrix(
    i = overlapTDF[, 1], 
    j = overlapTDF[, 4],
    x = rep(1, nrow(overlapTDF)), 
    dims = c(length(query), length(unique(overlapDF$name))))
  colnames(sparseM) <- unique(overlapDF$name)
  total <- total[colnames(sparseM)]
  frip <- frip[colnames(sparseM)]
  out <- list(counts = sparseM, frip = frip, total = total)
  return(out)
}

In [ ]:
## variables needed for funtions
genome <- BSgenome.Hsapiens.UCSC.hg38
chromSizes <- GRanges(names(seqlengths(genome)), IRanges(1, seqlengths(genome)))

In [ ]:
#### pass directory of peak calls 
dirPeaks = "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230117_peakCalling_postPromFilt_UCSC/peakCallOutput_qvalue05_UNparallelized_20230118"

In [ ]:
files = list.files(dirPeaks)[grepl("_summits.bed",list.files(dirPeaks) )]
files
clusters = gsub("_summits.bed","", files)
clusters


In [ ]:
df <- data.frame(
  samples = gsub("\\_summits.bed","",list.files(dirPeaks, pattern = "\\_summits.bed", full.names = FALSE)),
  groups = "snATAC",
  summits = list.files(dirPeaks, pattern = "\\_summits.bed", full.names = TRUE)
  )
#df <- df[-c(10), ]
(df)
#### below I removed celltypes that were too small to call peaks on like MUC5b in my dataset is like 4 cells
df2 <- df[!(row.names(df) %in% c("15")),]
(df2)


In [ ]:
rownames(df2)


### create cell type specific fixed peak file

In [ ]:
# This will create a fixed peak file for each cell type, basically it takes the summit of each peak in a celltype
# and extends it 250 each way so every peak is 500 bp
samples <- df2[,1]
for (j in 1:length(samples)) {
    df_test <- df2[j,]
    unionPeaks <- extendedPeakSet(
    df = df_test,
    BSgenome = genome, 
    extend = 250,
    blacklist = "/nfs/lab/rlmelton/npod/scripts/references/hg38-blacklist.v2.bed",
    nSummits = 200000
  )
unionPeaks <- unionPeaks[seqnames(unionPeaks) %in% paste0("chr",c(1:22,"X"))]
#unionPeaks <- keepSeqlevels(unionPeaks, paste0("chr",c(1:22,"X")))
#fragmentFiles <- list.files("data", pattern = ".rds", full.names = TRUE)
peakfile <- data.frame(unionPeaks)
df_peak = subset(peakfile, select = -c(width, strand, score, scoreQuantile) )
write.table(df_peak , file = sprintf("/nfs/lab/projects/nPOD/downstream_files/ATAC/pseudobulk_nPODids/fixed_peak/%s_fixedPeakFile.txt",samples[j]),quote = FALSE, col.names = FALSE, row.names = FALSE , sep = '\t')
}

### create merge fixed peak list

In [ ]:
unionPeaks <- extendedPeakSet(
    df = df2,
    BSgenome = genome, 
    extend = 250,
    blacklist = "/nfs/lab/rlmelton/npod/scripts/references/hg38-blacklist.v2.bed",
    nSummits = 200000
  )
unionPeaks <- unionPeaks[seqnames(unionPeaks) %in% paste0("chr",c(1:22,"X"))]
unionPeaks <- keepSeqlevels(unionPeaks, paste0("chr",c(1:22,"X")))


In [ ]:
peakfile <- data.frame(unionPeaks)
df = subset(peakfile, select = -c(width, strand, score, scoreQuantile) )


In [ ]:
write.table(df , file = "/nfs/lab/projects/nPOD/downstream_files/ATAC/pseudobulk_nPODids/fixed_peak/20230221_fixedPeakFile.txt",quote = FALSE, col.names = FALSE, row.names = FALSE , sep = '\t')

In [ ]:
df$merge <- paste(df$seqnames, ":", df$start,"-",df$start,sep = '')
head(df)

write.table(df$merge , file = "/nfs/lab/projects/nPOD/downstream_files/ATAC/pseudobulk_nPODids/fixed_peak/fixedPeakFile_formatted.txt",quote = FALSE, col.names = FALSE, row.names = FALSE , sep = '\t')


# Pseudo bulk using fixed peak list

Add fixed peak as an assay
Steps: 
- make lfm using fixed peak
- add as assay

### Run in bash
##### merge peak file /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/fixedPeakFile.txt
```bash
##### make count matrices by celltype using new merge peak list
#!/bin/bash
Pyscript_fp=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/Josh_10XPipeline_withPeaks_justLFM_fixedPeak.py
samples=(Acinar_1_2_6 Acinar_3 Acinar_5 Acinar_4 Activated_Stellate Endothelial Schwann Alpha Delta Beta Macrophage Tcells Mast Quiescent_Stellate LymphEndo Bcells MUC5b_Ductal Ductal)
N=5
for sample in ${samples[@]}; do
        ((i=i%N)); ((i++==0)) && wait
        (
                 outs_dir=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/fixedPeak_lfm/${sample}/
                mkdir $outs_dir
                keep_dir=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230117_peakCalling_postPromFilt_UCSC/${sample}

                tagAlign=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230117_peakCalling_postPromFilt_UCSC/finalTags/${sample}

                /usr/bin/python3 $Pyscript_fp  -o $outs_dir -k ${keep_dir}_barcodes.txt -n ${sample} -a ${tagAlign}.filt.rmdup.2023-01-17.broad.tagAlign.gz -t 24 -m 2 > ${outs_dir}log_file.txt
        ) &
done

```

In [ ]:
#### READ IN JOSHS MATRICES and then I have a ghetto work around because I'm dumb in R
#read in ATAC data from the lfm matrices (sm workaround method for now)
# load in starting ATAC long format matrices to a list(CellRanger filtered for now)
atacs_OG <- list()
atacs_FINAL <- list()
samples <- c('Acinar_1_2_6',  'Acinar_3' , 'Acinar_4' , 'Acinar_5',
             'Activated_Stellate' , 'Alpha' , 'Bcells' , 'Beta' ,
             'Delta' , 'Ductal',  'Endothelial' , 'LymphEndo'  ,'Macrophage',
             'Mast' ,  'Quiescent_Stellate'  , 'Tcells','MUC5b_Ductal','Schwann')
for (sample in samples) {
    #print(sample)
    wd <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/March_correctFiles/lfm_matrices/%s/', sample)
    atacs_OG[[sample]] <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',sample)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    #atacs_OG[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V1)
    #atacs_OG[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V2)
    atacs_FINAL[[sample]] <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',sample)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    atacs_FINAL[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V2)
    atacs_FINAL[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V1)
    atacs_OG[[sample]] <- NULL
    #atacs_FINAL[[sample]] <- atacs_FINAL[[sample]][atacs_FINAL[[sample]]$V2 %in% good,]
}
atac_mod <- atacs_FINAL

atac_mod$Acinar_1_2_6$V2 <- paste0(atac_mod$Acinar_1_2_6$V2, "-1")
atac_mod$Acinar_3$V2 <- paste0(atac_mod$Acinar_3$V2, "-1")
atac_mod$Acinar_4$V2 <- paste0(atac_mod$Acinar_4$V2, "-1")
atac_mod$Acinar_5$V2 <- paste0(atac_mod$Acinar_5$V2, "-1")
atac_mod$Activated_Stellate$V2 <- paste0(atac_mod$Activated_Stellate$V2, "-1")
atac_mod$Alpha$V2 <- paste0(atac_mod$Alpha$V2, "-1")
atac_mod$Bcells$V2 <- paste0(atac_mod$Bcells$V2, "-1")
atac_mod$Beta$V2 <- paste0(atac_mod$Beta$V2, "-1")
atac_mod$Delta$V2 <- paste0(atac_mod$Delta$V2, "-1")
atac_mod$Ductal$V2 <- paste0(atac_mod$Ductal$V2, "-1")
atac_mod$Endothelial$V2 <- paste0(atac_mod$Endothelial$V2, "-1")
atac_mod$LymphEndo$V2 <- paste0(atac_mod$LymphEndo$V2, "-1")
atac_mod$Macrophage$V2 <- paste0(atac_mod$Macrophage$V2, "-1")
atac_mod$Mast$V2 <- paste0(atac_mod$Mast$V2, "-1")
atac_mod$MUC5b_Ductal$V2 <- paste0(atac_mod$MUC5b_Ductal$V2, "-1")
atac_mod$Quiescent_Stellate$V2 <- paste0(atac_mod$Quiescent_Stellate$V2, "-1")
atac_mod$Schwann$V2 <- paste0(atac_mod$Schwann$V2, "-1")
atac_mod$Tcells$V2 <- paste0(atac_mod$Tcells$V2, "-1")


###### Mod peak file 
```bash 
cat mergedPeak.txt | awk '{print $1 ":" $2 "-" $3}' | awk '{print substr($0,4)}' > mergedPeak_mod.txt
```

In [ ]:
### GRAB THE WINDOWS FILE FROM JOSH'S PIPELINE 
#read in the HVWs set, we'll cut down all samples to be these windows
hvw_fp2 <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/20230222_fixedPeak_mod.txt'
hvws2 <- scan(hvw_fp2, what="", sep="\n")
print(head(hvws2))

#sort alphanumerically
hvws_fin2 <- sort(hvws2)
print(head(hvws_fin2))

In [ ]:
head(atac_mod)

In [ ]:
overall_sm2 <- merge_sparse_matrices_hvws(atac_mod,hvws_fin2)
dim(overall_sm2)
head(overall_sm2)

#check if the windows of overall_sm are sorted (they should be)
sorted_windows2 <- sort(row.names(overall_sm2))
table(sorted_windows2 == row.names(overall_sm2))

In [ ]:
#atac <- readRDS('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/20230119_snATAC_finalandFixedPeaksIncl.rds')
atac

In [ ]:
atac_mtx <- overall_sm2
### create new assay with TSS region matrix
frag.file ='/nfs/lab/rlmelton/npod/integration/nov18_seuratIntegration/fragment_files/merged.atac_fragments_test.tsv.gz'
atac_assay <- CreateChromatinAssay(counts=atac_mtx, sep=c(':', '-'), genome='hg38', fragments=frag.file, min.cells=0, min.features=-1)#, annotation=annotations)
atac[["FixPeaks_031023"]] <- atac_assay


In [ ]:
final_path <- paste0(fin_dir,'20230515_snATAC_finalandFixedPeaksIncl_correctedFixPeak.rds',sep='' )
saveRDS(atac, file = final_path)

In [ ]:
fin_dir

### create merged fixed peak psuedobulk count matrices

In [ ]:
#looping through cell types by making ^ into a function

get_per_sample_gex_SUMS <- function(cell.type, mtx.fp){
    print(paste(cell.type,Sys.time()))

    #pull out rows of gex.counts where BC Ident matches cell.type
    bcs <- names(Idents(adata_matrices)[Idents(adata_matrices) == cell.type])
    counts <- gex.counts[,colnames(gex.counts) %in% bcs]
    print(dim(counts))

    #initialize the matrix of sample gex
    counts.df <- as.data.frame(rep(0,length(row.names(gex.counts))))
    row.names(counts.df) <- row.names(gex.counts)
    colnames(counts.df) <- c('temp')

    #go through samples and calculate sum of gex values
    for (sample in samples){
        sample_cols <- colnames(counts) %in% sample_bcs[[sample]]
        counts.cut <- counts[,sample_cols]
        
        #if only one bc, this becomes a vector which is an issue
        if (typeof(counts.cut) == 'double'){
            mean.counts <- counts.cut
        #if there are NO bcs, this will return NA (just return 0 for everything)
        } else if(length(colnames(counts.cut)) == 0){
            mean.counts <- rep(0,length(row.names(counts)))
        } else {
            mean.counts <- rowSums(counts.cut)
        }
        counts.df <- cbind(counts.df,as.data.frame(mean.counts))
     }
    fin.counts.df <- counts.df[,-c(1)]
    colnames(fin.counts.df) <- samples
    head(fin.counts.df)

    #export df
    #mtx.fp <- sprintf('/nfs/lab/projects/multiomic_islet/outputs/multiome/rna_profiles/eQTL_matrices/%s_sample_gex_total_counts.txt',cell.type)
    write.table(fin.counts.df,mtx.fp,sep='\t',quote=FALSE)
}
######## SET TO WHATEVER YOUR ASSIGNMENTS ARE STORED UNDER ########
Idents(object = atac) <- "FinalSubtypes"
head(Idents(atac))
#### OUTPUT DIRECTORY #####
outdir = '/nfs/lab/projects/nPOD/downstream_files/ATAC/pseudobulk_nPODids/fixed_peak/merge_fixedPeak_mtx/pseudobulk_matrices/'
dir.create(outdir)

###### LIST SAMPLES ########
#samples <- c('MM_339','MM_340','MM_341','MM_342',
#           'MM_343', 'MM_344', 'MM_345','MM_365',
#           'MM_366','MM_367', 'MM_371', 'MM_383',
#           'MM_384','MM_385','MM_386','MM_387',
#           'MM_388','MM_390', 'MM_391','MM_392',
#           'MM_394','MM_395','MM_396','MM_460',
#           'MM_535','MM_536','MM_537','MM_543',
#           'MM_544','MM_545','MM_546','MM_547',
#           'MM_660','MM_661','MM_662','MM_664',
#           'MM_665','MM_666','MM_667','6229_sort')
#
samples <- unique(atac$nPOD_ID)
#pull out list of all cell types, removing ignore
unique_cell_types <- unique(atac$FinalSubtypes)
unique_cell_types <- unique_cell_types[unique_cell_types != "MUC5b_Ductal"];
print(unique_cell_types)

################## THIS IS IF YOU WANT TO ONLY RETAIN SAMPLES WITH > 20 CELLS FOR THAT CELL TYPE, IT WILL REPLACE THOSE WITH LESS WITH 0S #############
sample_bcs <- list()

for (sample in samples){
    print(sample)
    good_celltypes<- list()
    obj <- subset(atac, subset= nPOD_ID == sample)
    obj
    for (celltype in unique_cell_types){
        if (sum(Idents(obj)==celltype) >= 20){
            good_celltypes[[celltype]] <- celltype #row.names(obj[[]][obj[[]]$final_assignments == celltype,])
        }
        good <- names(good_celltypes)
    sample_bcs[[sample]] <- row.names(obj[[]][obj[[]]$FinalSubtypes %in% good,])
    }
}

################ IF YOU DO NOT WANT TO FILTER BY A MIN CELL COUNT NUMBER UNCOMMENT THE CODE BELOW########
#sample_bcs <- list()
#for (sample in samples){
#    sample_bcs[[sample]] <- row.names(soupX_obj[[]][soupX_obj[[]]$samples == sample,])
#}
#
#lengths(sample_bcs)
#head(sample_bcs[[1]])

##############
#### SET TO WHATEVER ASSAY YOU WANT TO USE ######
DefaultAssay(atac) <- 'FixPeaks_031023'
gex.counts <- GetAssayData(atac,slot='counts')
dim(gex.counts)
head(gex.counts)
adata_matrices <- atac


##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    fp = sprintf('/nfs/lab/projects/nPOD/downstream_files/ATAC/pseudobulk_nPODids/fixed_peak/merge_fixedPeak_mtx/pseudobulk_matrices/%s_pseudobulk_matrix_fixedPeak.txt',cell.type)
    get_per_sample_gex_SUMS(cell.type, fp)
}

# ATAC peak info file

```bash
cells=(Acinar_1_2_6 Acinar_3 Acinar_5 Acinar_4 Activated_Stellate Endothelial Schwann Alpha Delta Beta Macrophage Tcells Mast Quiescent_Stellate LymphEndo Bcells MUC5b_Ductal Ductal)

        for cell in ${cells[@]}; do bedtools intersect -a  /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/fixedPeakFile.txt -b /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230117_peakCalling_postPromFilt_UCSC/peakCallOutput_qvalue05_UNparallelized_20230118/${cell}_summits.bed -wa -wb > ${cell}_intersect.txt; done


cat *_intersect.txt |  awk '{print $1":" $2 "-" $3, $9}' > intersected_celltype.txt
```

In [ ]:
fix_dir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/' 


In [ ]:
atacs_OG <- read.table(file.path(fix_dir, "intersected_celltype_20230222.txt"), sep=' ', header=FALSE, stringsAsFactors=FALSE)
atacs_FINAL <- read.table(file.path(fix_dir, "intersected_celltype_20230222.txt"), sep=' ', header=FALSE, stringsAsFactors=FALSE)
atacs_FINAL$V2 <- as.factor(atacs_OG$V2)
atacs_FINAL$V1 <- as.factor(atacs_OG$V1)
head(atacs_FINAL)
atacs_FINAL$V3 <- 1

In [ ]:
hvw_fp2 <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/20230222_fixedPeak_mod.txt'
hvws2 <- scan(hvw_fp2, what="", sep="\n")
print(head(hvws2))

#sort alphanumerically
hvws_fin2 <- sort(hvws2)
print(head(hvws_fin2))

In [ ]:
head(atacs_FINAL)

In [ ]:
overall_sm <- with(atacs_FINAL,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
print(dim(overall_sm))

In [ ]:
mat <- as.matrix(overall_sm)
df_new <- as.data.frame(mat)
#colnames(df_new) = c('Acinar','Activated_stellate','Alpha','Bcells',
#                     'Beta','Delta','Ductal','Endothelial','LymphEndo',
#                     'Macrophage','Mast','Quiescent_stellate','Schwann','Tcells')
#

In [ ]:
head(df_new)

In [ ]:
dim(df_new)

In [ ]:
cm <- colSums(df_new)
cm

In [ ]:
write.table(df_new, file ="/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/peakInfo_wSubtypes_20230222.txt", quote = FALSE, row.names = TRUE, col.names = TRUE)


# ATAC cell count UMAP file

In [ ]:
atac

In [ ]:
UMAP <-table(atac@meta.data$FinalSubtypes,atac@meta.data$library)
write.table(UMAP, file= '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/cellcounts.txt', sep = '\t', quote = FALSE)


# ATAC metadata 

In [ ]:
colnames(atac@meta.data)


In [ ]:
### make metadata file with cell type proportion and counts 
#rna <- readRDS('/nfs/lab/rlmelton/npod/FinalFiles/22November/snRNA_object_20221208_subtypesLabelled_RAWandSOUP_multiAssayObj.rds')
#rna <- readRDS('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/Nov22_objects_final/20221216_FINAL_RNA.rds')
atac
head(atac@meta.data)

cellsubtype <- unique(atac@meta.data$FinalSubtypes)
sample <- unique(atac@meta.data$library)
cell_type <- list()


for (CELL in cellsubtype){
    gene_number <- list()
    for (SAMPLE in sample){
    
    gene_number[SAMPLE] <- mean(atac@meta.data$frac_reads_in_peaks[atac@meta.data$library == SAMPLE & atac@meta.data$FinalSubtypes == CELL])
}
    cell = paste0(CELL,'_meanFRiP')
 cell_type[[cell]] <- gene_number
}
mean_gene <- as.data.frame(do.call(cbind, cell_type))
mean_matrix <- as.matrix(mean_gene)
mean_matrix[sapply(mean_matrix, is.nan)] <- 0
mean_df <- as.data.frame(mean_matrix)
head(mean_df)
dim(mean_df)

count_original =read.table('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/cellcounts.txt', header = TRUE, sep='\t')
#count_original
count_t <- as.data.frame(t(count_original))
count_t[,"Total" ] <- rowSums(count_t)

for (CELL in cellsubtype) {
    column_name = paste0(CELL,"_proportion")
    #print(column_name)
    count_t[,column_name] <- count_t[,CELL]/count_t[,"Total" ]  
}

count_t$SeqID.ATAC <- row.names(count_t)   
mean_df$SeqID.ATAC <- row.names(mean_df)                     # Apply row.names function
# Apply row.names function
df = merge(x=count_t,y=mean_df,by="SeqID.ATAC")
head(df)

wd_meta='/nfs/lab/rlmelton/npod/'
metadata_original =read.table(paste0(wd_meta,'nPOD_clinical_July2022_metadata.txt'), header = TRUE, sep='\t')
head(metadata_original)

df2 = merge(x=metadata_original,y=df,by="SeqID.ATAC")
head(df2)

df2 = as.matrix(df2)
df2
write.table(df2, file='/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/nPOD_snATAC_cellCount_subtypes_wProportions_wMeanFRiP_20230120.txt', quote = FALSE, sep = '\t')



# Make CELL TYPE SPECIFIC fixed peak pseudo bulk matrices

##### merge peak file /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/fixedPeakFile.txt
```bash
##### make count matrices by celltype using new merge peak list
#!/bin/bash
Pyscript_fp=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/Josh_10XPipeline_withPeaks_justLFM_fixedPeak_modIndWindowFile.py
samples=(Acinar_1_2_6 Acinar_3 Acinar_5 Acinar_4 Activated_Stellate Endothelial Schwann Alpha Delta Beta Macrophage Tcells Mast Quiescent_Stellate LymphEndo Bcells MUC5b_Ductal Ductal)
#samples=(Beta)
N=5
for sample in ${samples[@]}; do
        ((i=i%N)); ((i++==0)) && wait
        (
                    outs_dir=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/celltypeSpecificFixedPeakLFM/${sample}/
                mkdir $outs_dir
                keep_dir=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230117_peakCalling_postPromFilt_UCSC/${sample}

                tagAlign=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230117_peakCalling_postPromFilt_UCSC/finalTags/${sample}
 window=/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/${sample}

                /usr/bin/python3 $Pyscript_fp  -o $outs_dir -k ${keep_dir}_barcodes.txt -n ${sample} -a ${tagAlign}.filt.rmdup.2023-01-17.broad.tagAlign.gz -w ${window}_fixedPeakFile.txt -t 24 -m 2 > ${outs_dir}log_file.txt
        ) &
done

```

In [ ]:
atac <- readRDS('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/snATAC/FilterATAC_byPromoterAccessibility/20230118_snATAC_filtPromUCSCcoords.rds')

In [ ]:
Idents(object = atac) <- "FinalSubtypes"
#adata <- RenameIdents(adata, final_assignments)
head(Idents(atac))

In [ ]:
#looping through cell types by making ^ into a function

get_per_sample_gex_SUMS_fixedPeak <- function(cell.type, mtx.fp){
    print(paste(cell.type,Sys.time()))
    wd <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/celltypeSpecificFixedPeakLFM/%s/', cell.type)
    atacs_OG <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',cell.type)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    #atacs_OG[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V1)
    #atacs_OG[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V2)
    atacs_FINAL <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',cell.type)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    atacs_FINAL$V1 <- as.factor(atacs_OG$V2)
    atacs_FINAL$V2 <- as.factor(atacs_OG$V1)
    overall_sm <- with(atacs_FINAL,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
    colnames(overall_sm) <- paste0(colnames(overall_sm), "-1")

    gex.counts <- overall_sm
    #pull out rows of gex.counts where BC Ident matches cell.type
    bcs <- names(Idents(atac)[Idents(atac) == cell.type])
    #bcs <- test.cell
    counts <- gex.counts[,colnames(gex.counts) %in% bcs]
    print(dim(counts))

    #initialize the matrix of sample gex
    counts.df <- as.data.frame(rep(0,length(row.names(gex.counts))))
    row.names(counts.df) <- row.names(gex.counts)
    colnames(counts.df) <- c('temp')

    #go through samples and calculate sum of gex values
    for (sample in samples){
        sample_cols <- colnames(counts) %in% sample_bcs[[sample]]
        counts.cut <- counts[,sample_cols]
        
        #if only one bc, this becomes a vector which is an issue
        if (typeof(counts.cut) == 'double'){
            mean.counts <- counts.cut
        #if there are NO bcs, this will return NA (just return 0 for everything)
        } else if(length(colnames(counts.cut)) == 0){
            mean.counts <- rep(0,length(row.names(counts)))
        } else {
            mean.counts <- rowSums(counts.cut)
        }
        counts.df <- cbind(counts.df,as.data.frame(mean.counts))
     }
    fin.counts.df <- counts.df[,-c(1)]
    colnames(fin.counts.df) <- samples
    head(fin.counts.df)

    #export df
    #mtx.fp <- sprintf('/nfs/lab/projects/multiomic_islet/outputs/multiome/rna_profiles/eQTL_matrices/%s_sample_gex_total_counts.txt',cell.type)
    write.table(fin.counts.df,mtx.fp,sep='\t',quote=FALSE)
}

In [ ]:
unique_cell_types <- unique(atac$FinalSubtypes)
#unique_cell_types <- unique_cell_types[unique_cell_types != "MUC5b_Ductal"];
unique_cell_types

In [ ]:
samples <- c('MM_339','MM_340','MM_341','MM_342',
           'MM_343', 'MM_344', 'MM_345','MM_365',
           'MM_366','MM_367', 'MM_371', 'MM_383',
           'MM_384','MM_385','MM_386','MM_387',
           'MM_388','MM_390', 'MM_391','MM_392',
           'MM_394','MM_395','MM_396','MM_460',
           'MM_535','MM_536','MM_537','MM_543',
           'MM_544','MM_545','MM_546','MM_547',
           'MM_660','MM_661','MM_662','MM_664',
           'MM_665','MM_666','MM_667','6229_sort')
sample_bcs <- list()

for (sample in samples){
    print(sample)
    good_celltypes<- list()
    obj <- subset(atac, subset= library == sample)
    obj
    for (celltype in unique_cell_types){
        if (sum(Idents(obj)==celltype) >= 20){ ### or >= 0 if you want all cells 
            good_celltypes[[celltype]] <- celltype #row.names(obj[[]][obj[[]]$final_assignments == celltype,])
        }
        good <- names(good_celltypes)
    sample_bcs[[sample]] <- row.names(obj[[]][obj[[]]$FinalSubtypes %in% good,])
    }
}

In [ ]:
##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    fp = sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/celltypeSpecificFixedPeakLFM/%s_pseudobulk_matrix_CellTypeSpecific_FixedPeak.txt',cell.type)
    get_per_sample_gex_SUMS_fixedPeak(cell.type, fp)
}

### extra code

In [ ]:
cell.type#head(overall_sm)

In [ ]:
head(sample_bcs)

In [ ]:
######## SET TO WHATEVER YOUR ASSIGNMENTS ARE STORED UNDER ########
Idents(object = atac) <- "FinalSubtypes"
head(Idents(atac))
#### OUTPUT DIRECTORY #####
outdir = '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/celltypeSpecificFixedPeakLFM/'
dir.create(outdir)

###### LIST SAMPLES ########
samples <- c('MM_339','MM_340','MM_341','MM_342',
           'MM_343', 'MM_344', 'MM_345','MM_365',
           'MM_366','MM_367', 'MM_371', 'MM_383',
           'MM_384','MM_385','MM_386','MM_387',
           'MM_388','MM_390', 'MM_391','MM_392',
           'MM_394','MM_395','MM_396','MM_460',
           'MM_535','MM_536','MM_537','MM_543',
           'MM_544','MM_545','MM_546','MM_547',
           'MM_660','MM_661','MM_662','MM_664',
           'MM_665','MM_666','MM_667','6229_sort')


#pull out list of all cell types, removing ignore
unique_cell_types <- unique(atac$FinalSubtypes)
#unique_cell_types <- unique_cell_types[unique_cell_types != "MUC5b_Ductal"];
print(unique_cell_types)

################## THIS IS IF YOU WANT TO ONLY RETAIN SAMPLES WITH > 20 CELLS FOR THAT CELL TYPE, IT WILL REPLACE THOSE WITH LESS WITH 0S #############
sample_bcs <- list()

for (sample in samples){
    print(sample)
    good_celltypes<- list()
    obj <- subset(atac, subset= library == sample)
    obj
    for (celltype in unique_cell_types){
        if (sum(Idents(obj)==celltype) >= 20){
            good_celltypes[[celltype]] <- celltype #row.names(obj[[]][obj[[]]$final_assignments == celltype,])
        }
        good <- names(good_celltypes)
    sample_bcs[[sample]] <- row.names(obj[[]][obj[[]]$FinalSubtypes %in% good,])
    }
}

################ IF YOU DO NOT WANT TO FILTER BY A MIN CELL COUNT NUMBER UNCOMMENT THE CODE BELOW########
#sample_bcs <- list()
#for (sample in samples){
#    sample_bcs[[sample]] <- row.names(soupX_obj[[]][soupX_obj[[]]$samples == sample,])
#}
#
#lengths(sample_bcs)
#head(sample_bcs[[1]])

##############



In [ ]:
cell.type <- 'Beta'

wd <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/celltypeSpecificFixedPeakLFM/%s/', cell.type)
atacs_OG <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',cell.type)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
#atacs_OG[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V1)
#atacs_OG[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V2)
atacs_FINAL <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',cell.type)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
atacs_FINAL$V1 <- as.factor(atacs_OG$V2)
atacs_FINAL$V2 <- as.factor(atacs_OG$V1)
overall_sm <- with(atacs_FINAL,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
colnames(overall_sm) <- paste0(colnames(overall_sm), "-1")

gex.counts <- overall_sm
#pull out rows of gex.counts where BC Ident matches cell.type
bcs <- names(Idents(atac)[Idents(atac) == cell.type])
#bcs <- test.cell
counts <- gex.counts[,colnames(gex.counts) %in% bcs]
print(dim(counts))

#initialize the matrix of sample gex
counts.df <- as.data.frame(rep(0,length(row.names(gex.counts))))
row.names(counts.df) <- row.names(gex.counts)
colnames(counts.df) <- c('temp')

#go through samples and calculate sum of gex values
for (sample in samples){
    sample_cols <- colnames(counts) %in% sample_bcs[[sample]]
    counts.cut <- counts[,sample_cols]
    
    #if only one bc, this becomes a vector which is an issue
    if (typeof(counts.cut) == 'double'){
        mean.counts <- counts.cut
    #if there are NO bcs, this will return NA (just return 0 for everything)
    } else if(length(colnames(counts.cut)) == 0){
        mean.counts <- rep(0,length(row.names(counts)))
    } else {
        mean.counts <- rowSums(counts.cut)
    }
    counts.df <- cbind(counts.df,as.data.frame(mean.counts))
 }
fin.counts.df <- counts.df[,-c(1)]
colnames(fin.counts.df) <- samples
head(fin.counts.df)

In [ ]:
fp = sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/snATAC_final/fixed_peaks/celltypeSpecificFixedPeakLFM/%s_pseudobulk_matrix_CellTypeSpecific_FixedPeak.txt',cell.type)
get_per_sample_gex_SUMS_fixedPeak(cell.type, fp)